# 4. Evaluate

Avalia `eval_datasets` do experimento e salva `test_metrics.json` no run.

Pesos: run de treino do `experiment_id` (ou `cfg['weights']` quando não houver run).

## Preparação

In [ ]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from src import config, datasets, experiments, eval as eval_mod

## Configuração

In [ ]:
# Experimento em `experiments/` (mesmo nome usado no treino)
EXPERIMENT = "e2-all"

N_WANDB_SAMPLE_PREDICTIONS = 10   # 0 desativa
WEIGHTS_OVERRIDE           = ""   # vazio = usa pesos do run

In [ ]:
# Opcional: config completa inline
CONFIG = None

# CONFIG = {
#     "experiment_id": "e2-all",
#     "strategy": "direct",
#     "weights": "yolo11m.pt",
#     "dataset_stage": "interim",
#     "train_datasets": ["passenger-detection-bus", "inside-bus-view", "passenger-deakin"],
#     "eval_datasets":  ["passenger-detection-bus", "inside-bus-view", "passenger-deakin"],
#     "train_config": {
#         "epochs": 100, "patience": 20, "batch": 16, "imgsz": 640,
#         "workers": 4, "amp": True, "seed": 42, "deterministic": True,
#     },
# }

## Configuração resolvida

In [ ]:
cfg = experiments.resolve(EXPERIMENT, CONFIG)

print("Experimentos disponíveis:", experiments.available())
print("Datasets disponíveis:    ", datasets.available(stage=cfg.get("dataset_stage", "interim")))
cfg

## Avaliação

Executa `eval.run_experiment(cfg, ...)`.

In [ ]:
metrics = eval_mod.run_experiment(
    cfg,
    n_wandb_samples=N_WANDB_SAMPLE_PREDICTIONS,
    weights_override=WEIGHTS_OVERRIDE,
)

## Resumo

In [ ]:
print(f"{'dataset':28s} {'mAP50':>8s} {'mAP50-95':>10s} {'precision':>10s} {'recall':>8s}")
for ds, m in metrics.items():
    print(
        f"{ds:28s}"
        f" {m.get('metrics/mAP50(B)', float('nan')):8.4f}"
        f" {m.get('metrics/mAP50-95(B)', float('nan')):10.4f}"
        f" {m.get('metrics/precision(B)', float('nan')):10.4f}"
        f" {m.get('metrics/recall(B)', float('nan')):8.4f}"
    )